## **AdaBoost: Step-by-Step Working**

### **Topic Roadmap**

**1. Prepare a small binary dataset**

**2. Initialize sample weights**

**3. Fit weak learners and calculate alpha**

**4. Update and normalize weights**

**5. Aggregate weighted predictions**

**6. Key revision notes**

## **1. Small Training Dataset**

AdaBoost focuses later learners on samples misclassified by earlier learners. Labels are encoded as `-1` and `+1` because the update rule uses signed predictions.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

X = np.array([[1, 5], [2, 3], [3, 6], [4, 8], [5, 1], [6, 9], [6, 5], [7, 8], [9, 9], [9, 2]], dtype=float)
y = np.array([1, 1, -1, 1, -1, 1, -1, 1, -1, -1])
weights = np.full(y.shape[0], 1 / y.shape[0])
summary = pd.DataFrame(X, columns=["x1", "x2"])
summary["y"] = y
summary["weight_0"] = weights
summary

,x1,x2,y,weight_0
0,1.0,5.0,1,0.1
1,2.0,3.0,1,0.1
2,3.0,6.0,-1,0.1
3,4.0,8.0,1,0.1
4,5.0,1.0,-1,0.1
5,6.0,9.0,1,0.1
6,6.0,5.0,-1,0.1
7,7.0,8.0,1,0.1
8,9.0,9.0,-1,0.1
9,9.0,2.0,-1,0.1


## **2. Train One Weak Learner**

A depth-one tree is a decision stump. Its weighted error determines its contribution to the final ensemble.

In [7]:
def fit_stump(X, y, sample_weight):
    stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
    stump.fit(X, (y == 1).astype(int), sample_weight=sample_weight)
    signed_pred = np.where(stump.predict(X) == 1, 1, -1)
    error = np.sum(sample_weight[signed_pred != y])
    error = np.clip(error, 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - error) / error)
    return stump, signed_pred, error, alpha

In [8]:
RANDOM_STATE = 42
stump_1, pred_1, error_1, alpha_1 = fit_stump(X, y, weights)
weights_1 = weights * np.exp(-alpha_1 * y * pred_1)
weights_1 /= weights_1.sum()
print(f"Weighted error: {error_1:.3f}; alpha: {alpha_1:.3f}")

Weighted error: 0.300; alpha: 0.424


## **3. Repeat with Updated Weights**

Misclassified rows receive larger normalized weights, making them more influential for the next stump.

In [9]:
stump_2, pred_2, error_2, alpha_2 = fit_stump(X, y, weights_1)
weights_2 = weights_1 * np.exp(-alpha_2 * y * pred_2)
weights_2 /= weights_2.sum()
weight_table = summary[["x1", "x2", "y"]].copy()
weight_table["weight_after_stump_1"] = weights_1
weight_table["weight_after_stump_2"] = weights_2
weight_table

,x1,x2,y,weight_after_stump_1,weight_after_stump_2
0,1.0,5.0,1,0.071429,0.166667
1,2.0,3.0,1,0.071429,0.166667
2,3.0,6.0,-1,0.071429,0.045455
3,4.0,8.0,1,0.166667,0.106061
4,5.0,1.0,-1,0.071429,0.045455
5,6.0,9.0,1,0.166667,0.106061
6,6.0,5.0,-1,0.071429,0.045455
7,7.0,8.0,1,0.166667,0.106061
8,9.0,9.0,-1,0.071429,0.166667
9,9.0,2.0,-1,0.071429,0.045455


## **4. Weighted Ensemble Prediction**

The final signed score is the sum of each weak learner prediction multiplied by its alpha. The sign of this score is the ensemble class.

In [10]:
alphas = np.array([alpha_1, alpha_2])
stumps = [stump_1, stump_2]
def predict_boosted(X_new):
    signed_predictions = np.column_stack([np.where(stump.predict(X_new) == 1, 1, -1) for stump in stumps])
    return np.sign(signed_predictions @ alphas)

ensemble_pred = predict_boosted(X)
print("Training accuracy:", np.mean(ensemble_pred == y))

Training accuracy: 0.7


### **Key Revision Notes**

- AdaBoost starts with equal sample weights.
- A weak learner with error below 0.5 receives positive weight `alpha`.
- Misclassified observations gain relative weight after the update.
- The final prediction is a weighted vote, not an unweighted majority vote.